In [2]:
!pip install "transformers==4.44.2" "huggingface_hub==0.23.5" --upgrade
#!pip install torch==2.9.1 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu128

  Using cached huggingface_hub-0.23.5-py3-none-any.whl.metadata (12 kB)
Using cached huggingface_hub-0.23.5-py3-none-any.whl (402 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.1.5
    Uninstalling huggingface_hub-1.1.5:
      Successfully uninstalled huggingface_hub-1.1.5


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.4.1 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 0.23.5 which is incompatible.


In [4]:
!pip install evaluate accelerate

  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached huggingface_hub-1.1.5-py3-none-any.whl.metadata (13 kB)
Using cached evaluate-0.4.6-py3-none-any.whl (84 kB)
Using cached huggingface_hub-1.1.5-py3-none-any.whl (516 kB)

  Attempting uninstall: huggingface-hub

    Found existing installation: huggingface-hub 0.23.5

    Uninstalling huggingface-hub-0.23.5:

      Successfully uninstalled huggingface-hub-0.23.5

   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   -----------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.19.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.1.5 which is incompatible.
transformers 4.44.2 requires huggingface-hub<1.0,>=0.23.2, but you have huggingface-hub 1.1.5 which is incompatible.


In [15]:
!pip install scikit-learn

  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   -------------------------------------- - 8.7/8.9 MB 59.5 MB/s eta 0:00:01
   ---------------------------------------- 8.9/8.9 MB 45.9 MB/s  0:00:00
Using cached joblib-1.5.2-py3-none-any.whl (308 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   ---

In [10]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))


2.9.1+cu128
True
12.8
NVIDIA GeForce RTX 5060 Laptop GPU


In [1]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import numpy as np
import evaluate
import torch


C:\Users\moham\anaconda3\envs\cuda-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("ag_news")

train_ds = dataset["train"]   # 120,000 samples
test_ds  = dataset["test"]    # 7,600 samples

num_labels = 4  # AG News has 4 categories
print(train_ds[0])


{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


### BERT - Model Training

### Tokenization and Data Preprocessing
We are doing the following transformations on the data:
Now each example has:
- input_ids: [101, 2023, 2003, ...] / after converting each word of the text into id's
- attention_mask: [1, 1, 1, 0, 0, ...] / essentially making only the words of the text to have the attention and paddings added to the text to become umimportant for the classification.
- label: 0–3

In [3]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

max_len = 200

def preprocess(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",  # fixed length = max_len
        max_length=max_len
    )

train_ds_enc = train_ds.map(preprocess, batched=True)
test_ds_enc  = test_ds.map(preprocess, batched=True)

In [4]:
train_ds_enc = train_ds_enc.remove_columns(["text"])
test_ds_enc  = test_ds_enc.remove_columns(["text"])

In [5]:
train_ds_enc.set_format("torch")
test_ds_enc.set_format("torch")

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)


In [13]:

training_args = TrainingArguments(
    output_dir="./bert_agnews_results",
    learning_rate=2e-5,
    per_device_train_batch_size=6,   # 512 is huge; will likely OOM
    per_device_eval_batch_size=6,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=100,
    fp16=torch.cuda.is_available(),   # only used if GPU & supported
)


In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_enc,
    eval_dataset=test_ds_enc,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

C:\Users\moham\AppData\Local\Temp\ipykernel_31748\1537370970.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
100,0.317900
200,0.301900
300,0.313800
400,0.434800
500,0.443600
600,0.379100
700,0.489300
800,0.452000
900,0.362800
1000,0.416700



KeyboardInterrupt

